### Question 1
Download a publicly available dataset of your choice (for example, a Flipkart product ratings dataset or Spotify song features) and split it into features (X) and target (y) for a regression problem.

A Zomato restaurant dining ratings dataset (`zomato_restaurant_ratings_500.csv`) is loaded for regression modeling. The feature matrix ($X$) consists of 8 operational, culinary, and engagement predictors (`cost_for_two`, `table_booking`, `online_delivery`, `votes`, `cuisines_count`, `photo_count`, `distance_km`, `menu_page_count`), and the target variable ($y$) is the continuous customer dining rating (`rating`). Features are standardized using `StandardScaler` to ensure $L1$ and $L2$ regularization penalties apply uniformly across all variables.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load the dataset from CSV
df = pd.read_csv('zomato_restaurant_ratings_500.csv')

# 2. Separate feature matrix (X) and continuous target (y)
feature_cols = [
    'cost_for_two', 'table_booking', 'online_delivery', 'votes', 
    'cuisines_count', 'photo_count', 'distance_km', 'menu_page_count'
]
target_col = 'rating'

X = df[feature_cols]
y = df[target_col]

# 3. Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# 4. Standardize features for regularized regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Loaded dataset shape: {df.shape}")
print(f"Feature matrix X shape: {X.shape} | Target y shape: {y.shape}")
print(f"Training set samples: {X_train.shape[0]} | Testing set samples: {X_test.shape[0]}")
display(df.head())

Loaded dataset shape: (500, 9)
Feature matrix X shape: (500, 8) | Target y shape: (500,)
Training set samples: 400 | Testing set samples: 100


,cost_for_two,table_booking,online_delivery,votes,cuisines_count,photo_count,distance_km,menu_page_count,rating
0,1250.0,0,0,2600.0,4,1029.0,1.3,14,4.07
1,2860.0,0,1,2401.0,7,857.0,5.7,2,4.36
2,2250.0,0,1,138.0,3,78.0,12.2,7,3.76
3,1880.0,1,1,1713.0,7,623.0,6.3,12,5.00
4,640.0,0,1,1907.0,4,718.0,14.4,5,4.03


### Question 2
Fit a Lasso regression model (L1 regularization) using scikit-learn's Lasso class on your dataset, and print out the coefficients for each feature.

Lasso regression minimizes the residual sum of squares subject to an $L1$ absolute-value penalty: $\mathcal{L}_{Lasso} = \text{RSS} + \alpha \sum_{j=1}^p |\beta_j|$. Because the $L1$ constraint creates sharp corners along the coordinate axes, optimization drives non-informative and redundant feature coefficients to exactly $0.0$, acting as an intrinsic feature selector.

In [2]:
from sklearn.linear_model import Lasso

# Fit Lasso regression with alpha=0.08
lasso_model = Lasso(alpha=0.08, random_state=42)
lasso_model.fit(X_train_scaled, y_train)

# Print feature coefficients
lasso_coefs = pd.DataFrame({
    'Feature': feature_cols,
    'Lasso Coefficient (L1)': lasso_model.coef_
})

print(f"Lasso Intercept: {lasso_model.intercept_:.4f}\n")
print("Lasso Regression Feature Coefficients:")
display(lasso_coefs)

Lasso Intercept: 4.4631

Lasso Regression Feature Coefficients:


,Feature,Lasso Coefficient (L1)
0,cost_for_two,0.000000
1,table_booking,0.084621
2,online_delivery,0.023144
3,votes,0.283545
4,cuisines_count,0.043544
5,photo_count,0.000000
6,distance_km,-0.000000
7,menu_page_count,0.000000


### Question 3
Fit a Ridge regression model (L2 regularization) using scikit-learn's Ridge class on the same dataset, and compare the coefficients to those from the Lasso model.

*(Hint: Look for features where Lasso sets coefficients exactly to zero but Ridge does not.)*

Ridge regression introduces an $L2$ squared-magnitude penalty: $\mathcal{L}_{Ridge} = \text{RSS} + \alpha \sum_{j=1}^p \beta_j^2$. Ridge penalizes large weights and stabilizes estimates in the presence of multicollinearity, but its spherical constraint region ensures coefficients asymptotically approach zero without ever reaching zero exactly. In contrast, Lasso zeros out uninformative predictors (`distance_km`, `menu_page_count`, and `cost_for_two`), producing a sparse and interpretable model.

In [3]:
from sklearn.linear_model import Ridge

# Fit Ridge regression with alpha=10.0
ridge_model = Ridge(alpha=10.0, random_state=42)
ridge_model.fit(X_train_scaled, y_train)

# Compare Lasso and Ridge coefficients side-by-side
comparison_df = pd.DataFrame({
    'Feature': feature_cols,
    'Lasso (L1)': lasso_model.coef_,
    'Ridge (L2)': ridge_model.coef_,
    'Lasso Zeroed?': np.isclose(lasso_model.coef_, 0.0)
})

display(comparison_df.round(4))
print("Observation: Notice how distance_km, menu_page_count, and cost_for_two are exactly 0.0 in Lasso, whereas Ridge retains non-zero weights for every feature.")

,Feature,Lasso (L1),Ridge (L2),Lasso Zeroed?
0,cost_for_two,0.0000,0.0456,True
1,table_booking,0.0846,0.1556,False
2,online_delivery,0.0231,0.0825,False
3,votes,0.2835,0.2205,False
4,cuisines_count,0.0435,0.1211,False
5,photo_count,0.0000,0.1367,True
6,distance_km,-0.0000,0.0014,True
7,menu_page_count,0.0000,0.0017,True


Observation: Notice how distance_km, menu_page_count, and cost_for_two are exactly 0.0 in Lasso, whereas Ridge retains non-zero weights for every feature.


### Question 4
Fit an ElasticNet regression model using scikit-learn's ElasticNet class, and observe how the coefficients differ from both Lasso and Ridge.

*(Hint: Try changing the l1_ratio parameter and see how it affects the results.)*

ElasticNet linearly combines both $L1$ and $L2$ penalties: $\alpha \left[ \rho \sum |\beta_j| + \frac{1-\rho}{2} \sum \beta_j^2 \right]$, where `l1_ratio` ($\rho$) governs the penalty balance. When $\rho = 1.0$, ElasticNet is identical to Lasso; when $\rho = 0.0$, it becomes Ridge. At $\rho = 0.5$, ElasticNet achieves feature sparsity while distributing weights across correlated feature groups (such as `votes` and `photo_count`) rather than arbitrarily selecting one.

In [4]:
from sklearn.linear_model import ElasticNet

# Fit ElasticNet with balanced l1_ratio=0.5
elastic_model = ElasticNet(alpha=0.08, l1_ratio=0.5, random_state=42)
elastic_model.fit(X_train_scaled, y_train)

# Compare all three models side by side
all_models_df = pd.DataFrame({
    'Feature': feature_cols,
    'Ridge (L2)': ridge_model.coef_,
    'ElasticNet (l1_ratio=0.5)': elastic_model.coef_,
    'Lasso (L1)': lasso_model.coef_
})

print("Comparison Across Regularization Techniques:")
display(all_models_df.round(4))

# Demonstrate the effect of varying l1_ratio
print("\nEffect of Varying l1_ratio in ElasticNet on Number of Non-Zero Coefficients:")
for r in [0.1, 0.3, 0.5, 0.7, 0.9]:
    en = ElasticNet(alpha=0.08, l1_ratio=r, random_state=42)
    en.fit(X_train_scaled, y_train)
    non_zeros = np.sum(~np.isclose(en.coef_, 0.0))
    print(f"  l1_ratio = {r:.1f} -> Non-zero features: {non_zeros} / {len(feature_cols)}")

Comparison Across Regularization Techniques:


,Feature,Ridge (L2),ElasticNet (l1_ratio=0.5),Lasso (L1)
0,cost_for_two,0.0456,0.0082,0.0000
1,table_booking,0.1556,0.1174,0.0846
2,online_delivery,0.0825,0.0539,0.0231
3,votes,0.2205,0.1913,0.2835
4,cuisines_count,0.1211,0.0799,0.0435
5,photo_count,0.1367,0.1244,0.0000
6,distance_km,0.0014,-0.0000,-0.0000
7,menu_page_count,0.0017,0.0000,0.0000



Effect of Varying l1_ratio in ElasticNet on Number of Non-Zero Coefficients:
  l1_ratio = 0.1 -> Non-zero features: 6 / 8
  l1_ratio = 0.3 -> Non-zero features: 6 / 8
  l1_ratio = 0.5 -> Non-zero features: 6 / 8
  l1_ratio = 0.7 -> Non-zero features: 5 / 8
  l1_ratio = 0.9 -> Non-zero features: 5 / 8


### Question 5
Suppose you want to build a feature selector for a Zomato-style restaurant rating predictor. Use Lasso regression to select the most important features from your dataset, and list which features are kept (non-zero coefficients) and which are dropped.

Lasso functions as an automated feature filter by shrinking uninformative or redundant predictor weights to zero. For the Zomato restaurant rating predictor, the retained features reflect operational attributes that directly drive dining quality and engagement (table booking availability, delivery options, cuisine variety, and customer reviews), while noise variables (`distance_km`, `menu_page_count`, and minor pricing variations) are eliminated.

In [5]:
# Extract kept and dropped features based on non-zero Lasso coefficients
kept_features = [feat for feat, coef in zip(feature_cols, lasso_model.coef_) if not np.isclose(coef, 0.0)]
dropped_features = [feat for feat, coef in zip(feature_cols, lasso_model.coef_) if np.isclose(coef, 0.0)]

feature_selection_summary = pd.DataFrame({
    'Feature': feature_cols,
    'Lasso Coefficient': lasso_model.coef_,
    'Selection Status': ['KEPT' if feat in kept_features else 'DROPPED' for feat in feature_cols]
}).sort_values('Lasso Coefficient', ascending=False)

print("=== Feature Selection Summary for Zomato Rating Predictor ===\n")
display(feature_selection_summary)

print(f"Total Features Analyzed: {len(feature_cols)}")
print(f"Kept Features ({len(kept_features)}):    {kept_features}")
print(f"Dropped Features ({len(dropped_features)}): {dropped_features}")

=== Feature Selection Summary for Zomato Rating Predictor ===



,Feature,Lasso Coefficient,Selection Status
3,votes,0.283545,KEPT
1,table_booking,0.084621,KEPT
4,cuisines_count,0.043544,KEPT
2,online_delivery,0.023144,KEPT
0,cost_for_two,0.000000,DROPPED
5,photo_count,0.000000,DROPPED
6,distance_km,-0.000000,DROPPED
7,menu_page_count,0.000000,DROPPED


Total Features Analyzed: 8
Kept Features (4):    ['table_booking', 'online_delivery', 'votes', 'cuisines_count']
Dropped Features (4): ['cost_for_two', 'photo_count', 'distance_km', 'menu_page_count']
